---
title: "Gebäudesektormodell – Analyse"
execute:
  echo: true
format:
  html:
    code-fold: true
jupyter: python3
---

# Gebäudesektormodell-Analyse {#sec-gebaeudesektormodell}

::: {.callout-warning title="Neue Flächenlieferung erforderlich"}
**Aktualisierungsbedarf · 09.11.2026**

Die bisherige 900-Zeilen-Excel ist noch keine vollständige Zuordnung für die lokale Typologie mit 43.200 Simulationen. Nach der Lieferung die Analyse ausführen, Bestände exportieren und Berichtstabellen prüfen. Hier liegen noch keine neuen fachlichen Ergebnisse vor.
:::

## 1. Zweck und Eingabedateien
Dieses Notebook verbindet eine **fertige Flächenzuordnung** mit den simulierten Energiekennwerten.
Es modelliert keine neuen Flächenanteile. Die Herleitung der Flächen gehört in das Notebook
Gebaeudesektor_Modellierung. Die neue vollständige Flächentabelle des Kollegen steht noch aus;
die aktuelle historische Excel reicht für die neue Typologie nicht aus.

Die Analyse liefert einen Bestand für „heute“ und einen für „2040“: jede Zeile enthält
Simulation, Merkmale, zugeordnete BGF und Energiekennwerte. Erst im Kalibrierungsdashboard
wird ausgewählt, welcher Anteil dieses Bestands virtuell nach PEQ bewertet wird.
Ein PEQ-Ziel von 20 % wird hier nicht mehr vorausgesetzt.

Alle fachlichen Rechenschritte stehen in diesem Notebook. Benötigt werden nur die üblichen
Python-Bibliotheken und die beiden Dateien im lokalen Ordner data.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_quarto.yml').exists())
BASE = ROOT / 'Berechnungen/gebaeudesektormodell'
PARQUET = BASE / 'data/scenarios.parquet'
INPUT = BASE / 'data/Building-Sector_Pool_1.xlsx'
OUT = BASE / 'outputs/analyse'
# Publizierte Ergebnisse vom 05.11. bleiben bis zur neuen Lieferung unverändert.
GEN = OUT / 'berichtstabellen'

typology = pd.read_parquet(PARQUET)
areas = pd.read_excel(INPUT, sheet_name='Strombed_estim')
mapping_input = pd.read_excel(INPUT, sheet_name='zuordnung_gfz', usecols='A:F')
display(pd.DataFrame({'Quelle':['Simulationen','Flächenzuordnung'], 'Zeilen':[len(typology),len(areas)]}))
display(areas.head())

# Einheiten: Kennwerte je m² NGF, Flächenlieferung in m² BGF.
# Aktueller Pool: einheitlich 0,8. Bei abweichenden Typen ausdrücklich anpassen.
r = 0.8
params = dict(jahr_pe_bewertung=None, single_target_missing_share=1,
              density_f_res=30.4, density_f_ed=26.4, density_dx=.15, density_cutoff=62.)
inputs = {'GFZ_Zuordnung':mapping_input}
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
provenance = {'simulation':str(PARQUET.relative_to(ROOT)), 'source_sha256':sha256(PARQUET),
              'input_sha256':sha256(INPUT)}
def markdown_table(frame, caption=None, identifier=None):
    """Format a small table without an optional tabulate dependency."""
    def fmt(v):
        if pd.isna(v): return 'offen'
        if isinstance(v,float): return f'{v:,.3f}'.replace(',','X').replace('.',',').replace('X','.')
        return str(v).replace('|',' / ').replace('\n',' ')
    result = '| '+' | '.join(map(str,frame.columns))+' |\n'
    result += '| '+' | '.join(['---']*len(frame.columns))+' |\n'
    result += '\n'.join('| '+' | '.join(fmt(v) for v in row)+' |' for row in frame.itertuples(index=False,name=None))
    if caption: result += '\n\n: '+caption+(f' {{#{identifier}}}' if identifier else '')+'\n'
    return result


## 2. Fertige Flächenzuordnung mit Simulationen verbinden {#sec-gebaeude-datenpruefung}
`Archetype` in der Flächendatei ist der vollständige Szenarioname aus PEExcel.
Die Verbindung geschieht exakt über diesen Namen und den Batch. Kein Kürzen auf Basisarchetypen,
keine Gleichverteilung über Wetter oder Masse und keine synthetische Ergänzung von PV0.

Drei kurze Kontrollen schützen die Bedeutung der Rechnung: eindeutige und vollständige
Zuordnung, unveränderte Flächensummen und übereinstimmende Flächeneinheiten. Fehlende
Kennwerte werden in der jeweiligen Bilanz als offen ausgewiesen.


In [ ]:
area_columns = ['Nationale BGF heute','Nationale BGF 2040']
areas = areas.rename(columns={'Archetype':'scenario_name'})
keys = ['batch','scenario_name'] if 'batch' in areas else ['scenario_name']
if 'batch' not in areas and not typology.scenario_name.is_unique:
    raise ValueError('Bitte batch ergänzen: Szenarionamen kommen in mehreren Läufen vor.')
missing = areas.loc[~pd.MultiIndex.from_frame(areas[keys]).isin(pd.MultiIndex.from_frame(typology[keys])),keys]
if not missing.empty:
    display(missing.head())
    raise ValueError('Vollständige neue Flächentabelle fehlt: Archetype muss exakt zum scenario_name im lokalen Parquet passen. Keine automatische Variantenaufteilung.')
optional = [c for c in ['Bestand','Neubau','Sanierung','Energetischer Standard'] if c in areas]
data = typology.merge(areas[keys + area_columns + optional], on=keys, how='left', validate='one_to_one')
if data[area_columns].isna().any().any() or (data[area_columns] < 0).any().any():
    raise ValueError('Jeder Typ benötigt eine nichtnegative Fläche je Stand; nicht verwendete Typen ausdrücklich mit null angeben.')
np.testing.assert_allclose(data[area_columns].sum(),areas[area_columns].sum())
np.testing.assert_allclose(data.NFA_total/data.GFA_total,r)

# Aktueller Namensaufbau: sieben Merkmale. Originalnamen bleiben unverändert.
dimensions = ['Geometrie','Huelle','Heizung','PV_Klasse','Flexibilitaet','Wetter','Masse']
parts = data.scenario_name.str.split('|',expand=True)
parts.columns = dimensions
for column in dimensions:
    data[column] = parts[column].str.strip()
data['Archetype'] = data.batch + '::' + data.scenario_name
for column in ['Bestand','Neubau','Sanierung']:
    if column not in data: data[column] = np.nan
if 'Energetischer Standard' not in data: data['Energetischer Standard'] = ''
data['PEQ_Anteil_2040'] = 0.  # Auswahl erfolgt erst im Dashboard.
data['BGF_2020_m2'] = np.nan
for year in ['heute','2040']:
    data['Flächenanteil '+year] = data['Nationale BGF '+year] / data['Nationale BGF '+year].sum()
original_count = len(data)
data['Herkunft'] = 'simuliert'
OUT.mkdir(parents=True,exist_ok=True)
GEN.mkdir(parents=True,exist_ok=True)
display(data[['Archetype',*dimensions,*area_columns]].head(12))


## 3. Flächenbilanz {#sec-gebaeude-flaechen}
Die Basisflächen bleiben je Geometrie/Hülle/Versorgung erhalten. NGF = BGF × 0,8;
dies entspricht auch NFA_total/GFA_total im aktuellen Pool. Statusanteile sind flächengewichtet.
2020 bleibt offen. Nullflächen sind explizit ausgeschlossen, fehlende Angaben bleiben unbekannt.


In [ ]:
area_cols = {'heute': 'Nationale BGF heute', '2040': 'Nationale BGF 2040', '2020': 'BGF_2020_m2'}
areas = []
for year, c in area_cols.items():
    areas.append({'Stand': year, 'BGF [Mio. m²]': data[c].sum(min_count=1) / 1000000.0, 'NGF [Mio. m²]': r * data[c].sum(min_count=1) / 1000000.0, 'positiv': int(data[c].gt(0).sum()), 'Null': int(data[c].eq(0).sum()), 'fehlend': int(data[c].isna().sum())})
area_summary = pd.DataFrame(areas)
display(area_summary)
for year, share in [('heute', 'Flächenanteil heute'), ('2040', 'Flächenanteil 2040')]:
    expected = data[area_cols[year]] / data[area_cols[year]].sum(min_count=1)
    print(year, 'max. Abweichung der gelieferten Anteilsfelder:', abs(data[share] - expected).max())
statussum = data[['Bestand', 'Neubau', 'Sanierung']].sum(axis=1, min_count=3)
status = pd.DataFrame([{'Zustand': c, 'BGF 2040 [Mio. m²]': (data[c] * data['Nationale BGF 2040']).sum(min_count=1) / 1000000.0} for c in ['Bestand', 'Neubau', 'Sanierung']])
display(status)
geometry_area = data.groupby('Geometrie')[['Nationale BGF heute', 'Nationale BGF 2040']].sum(min_count=1) / 1000000.0
display(geometry_area)
fig, ax = plt.subplots(figsize=(9, 4))
geometry_area.plot.bar(ax=ax, color=['#9aaebc', '#285c83'])
ax.set(ylabel='BGF [Mio. m²]', xlabel='Geometrische Klasse')
ax.tick_params(axis='x', rotation=40)
ax.legend(['Heute: Jahr offen', '2040'])
fig.tight_layout()
fig.savefig(OUT / 'flaechen.png', dpi=180)
plt.show()
area_summary.to_csv(OUT / 'flaechenaudit.csv', index=False)


## 4. Physische Strombilanz {#sec-gebaeude-strom}
Bruttobedarf folgt EUI_el_total ohne EUIev_el; Mobilität wird erst beim sektoralen Vergleich angesetzt.
PV_total umfasst direkten und flexiblen Verbrauch sowie Einspeisung. Die Flussprüfung
berücksichtigt deshalb PV_total_flex und VRGrid_to_building. Keine virtuellen Kontextwerte
werden als Erzeugung addiert. Die Anwendungssummen schließen direktes elektrisches Warmwasser ein.


In [ ]:
data['Strombedarf'] = data.EUI_el_total - data.EUIev_el
data['PV_Erzeugung'] = data.PV_total
data['Stromsaldo'] = data.PV_Erzeugung - data.Strombedarf
apps = pd.DataFrame({'Heizung': data.EUIh_el - data.EUIh_el_aux, 'Warmwasser': data.EUIdhw_el + data.EUIdhwdirect_el - data.Edhw_aux_el, 'Kühlung': data.EUIc_el - data.EUIc_el_aux, 'Lüftung': data.EUIv_el, 'Hilfs-/Allgemeinstrom': data.EUI_auxiliary + data.EUIh_el_aux + data.EUIc_el_aux + data.Edhw_aux_el, 'Beleuchtung': data.EUI_lighting, 'Geräte/IT': data.EUI_plugloads})

def aggregate(values, area):
    missing_area = area.isna().any()
    active = area.gt(0)
    covered = active & values.notna()
    known = (values[covered] * area[covered] * r).sum(min_count=1)
    missing = area[active & values.isna()].sum()
    return dict(kWh=known if not missing_area and missing == 0 else np.nan, bekannte_kWh=known, fehlende_BGF_m2=missing, Flächen_offen=missing_area)
energy_records = []
for year, c in area_cols.items():
    for metric in ['Strombedarf', 'PV_Erzeugung', 'Grid_to_building_min', 'PV_total_direct', 'PV_total_flex', 'VRGrid_to_building', 'PV_to_Egrid', 'Stromsaldo']:
        a = aggregate(data[metric], data[c])
        energy_records.append({'Stand': year, 'Größe': metric, 'kWh/(m² NGF a)': a['kWh'] / (data[c].sum(min_count=1) * r), 'GWh/a': a['kWh'] / 1000000.0, 'PJ/a': a['kWh'] * 3.6e-09, 'fehlende BGF [m²]': a['fehlende_BGF_m2'], 'Flächen offen': a['Flächen_offen']})
energy = pd.DataFrame(energy_records)
display(energy[energy.Stand.eq('2040')])
weighted_apps = apps.mul(data['Nationale BGF 2040'] * r, axis=0)
application_total = weighted_apps.sum(min_count=1) / 1000000.0
display(application_total.rename('GWh/a').to_frame())
by_geometry = weighted_apps.groupby(data.Geometrie).sum(min_count=1).div((data['Nationale BGF 2040'] * r).groupby(data.Geometrie).sum(), axis=0)
fig, ax = plt.subplots(figsize=(10, 4.5))
by_geometry.dropna(how='all').plot.bar(stacked=True, ax=ax, colormap='tab20')
ax.set(ylabel='Strombedarf [kWh/(m² NGF a)]', xlabel='Geometrische Klasse')
ax.tick_params(axis='x', rotation=40)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.22), ncol=4, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'strom_anwendungen.png', dpi=180)
plt.show()
geom_energy = data[['Strombedarf', 'PV_Erzeugung', 'Stromsaldo']].mul(data['Nationale BGF 2040'] * r, axis=0).groupby(data.Geometrie).sum(min_count=1) / 1000000.0
display(geom_energy)
energy.to_csv(OUT / 'strombilanz.csv', index=False)
by_geometry.to_csv(OUT / 'strom_anwendungen_spezifisch.csv')
geom_energy.to_csv(OUT / 'strombilanz_geometrie.csv')


## 5. Endenergiebilanz nach Energieträgern und Anwendungen {#sec-gebaeude-endenergie}

Die elektrische Heizung ist nicht die gesamte Wärmeversorgung. Die Quelle enthält Fernwärme, Erdgas, Biomasse und „other“. **Öl wird aus `EUI_other` nur für Schlüssel mit Heizsystem `Oel` herausgelöst**. Ein anderer „other“-Betrag bleibt sonstige thermische Endenergie. Die Anwendungsaufteilung stammt aus den zugehörigen `EUIh_*`, `EUIdhw_*` und `EUIc_other`-Spalten, nicht aus angenommenen Wirkungsgraden.

$$E_{i,y}=\sum_k A_{NGF,k,y}\,e_{i,k},\qquad E_{End,y}=E_{el,y}+\sum_{i\ne el}E_{i,y}.$$

Der Bruttostrombedarf enthält lokal genutzten PV-Strom. Die alternativ ausgewiesene externe Lieferung enthält nur Netzbezug plus thermische Energieträger. Umweltwärme ist in diesen gelieferten Trägerkennwerten nicht als zusätzliche Endenergielieferung enthalten; diese Abgrenzung ist beim Vergleich mit nationalen Statistiken zu beachten.

In [ ]:
carriers = pd.DataFrame({'Strom (Bruttobedarf)': data.Strombedarf, 'Fernwärme': data.EUI_district_heating, 'Gas': data.EUI_natural_gas, 'Biomasse': data.EUI_biomass, 'Öl': data.EUI_other.where(data.Heizung.eq('Oel'), 0), 'Sonstige thermisch': data.EUI_other.where(~data.Heizung.eq('Oel'), 0)})
thermal_apps = []
for carrier, suffix in [('Fernwärme', 'district_heating'), ('Gas', 'natural_gas'), ('Biomasse', 'biomass'), ('Sonstige inkl. Öl', 'other')]:
    comps = {'Heizung': data['EUIh_' + suffix], 'Warmwasser': data['EUIdhw_' + suffix]}
    if suffix == 'other':
        comps['Kühlung'] = data.EUIc_other
    for app, series in comps.items():
        thermal_apps.append({'Träger': carrier, 'Anwendung': app, 'GWh/a': aggregate(series, data['Nationale BGF 2040'])['kWh'] / 1000000.0})
display(pd.DataFrame(thermal_apps))
ee = []
for year, c in area_cols.items():
    for carrier in carriers:
        a = aggregate(carriers[carrier], data[c])
        ee.append({'Stand': year, 'Träger': carrier, 'kWh/(m² NGF a)': a['kWh'] / (data[c].sum(min_count=1) * r), 'GWh/a': a['kWh'] / 1000000.0, 'PJ/a': a['kWh'] * 3.6e-09})
endenergy = pd.DataFrame(ee)
display(endenergy[endenergy.Stand.eq('2040')])
endenergy.to_csv(OUT / 'endenergiebilanz.csv', index=False)
pd.DataFrame(thermal_apps).to_csv(OUT / 'thermische_anwendungen_2040.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 4))
endenergy[endenergy.Stand.eq('2040')].set_index('Träger')['PJ/a'].plot.barh(ax=ax, color=['#dfb42c', '#b85654', '#847d7a', '#527f4d', '#53545b', '#9b8c70'])
ax.set(xlabel='Jährliche Endenergie [PJ/a]', ylabel='')
fig.tight_layout()
fig.savefig(OUT / 'endenergie.png', dpi=180)
plt.show()


## 6. PE-Bewertung realer Import-/Exportflüsse {#sec-gebaeude-pe}
PEI_saldo_project = PEI_export − PEI_import. Netzbezug und flexible Bezüge sind Komponenten,
thermische Energieträger bleiben enthalten. Die Größen sind keine virtuellen Kontextbilanzen.
Für synthetisches PV0 bleiben PE-Kennwerte offen; daher wird keine vollständige nationale
PE-Summe behauptet. Die physische Stromrechnung ist davon unabhängig vollständig.


In [ ]:
data['PE_Saldo'] = data.PEI_saldo_project
pe_records = []
for year, c in area_cols.items():
    for metric in ['PEI_grid_import', 'PEI_flex_import', 'PEI_import', 'PEI_export', 'PE_Saldo']:
        a = aggregate(data[metric], data[c])
        pe_records.append({'Stand': year, 'Größe': metric, 'GWh_PE/a': a['kWh'] / 1000000.0, 'PJ_PE/a': a['kWh'] * 3.6e-09, 'bekannter Teil [PJ PE/a]': a['bekannte_kWh'] * 3.6e-09, 'fehlende BGF [m²]': a['fehlende_BGF_m2']})
primary = pd.DataFrame(pe_records)
display(primary[primary.Stand.eq('2040')])
implicit = []
for carrier, ee_col, pe_col in [('Netzstrom', 'Grid_to_building_min', 'PEI_grid_import'), ('PV-Export', 'PV_to_Egrid', 'PEI_export'), ('Fernwärme', 'EUI_district_heating', 'PEI_district_heating'), ('Gas', 'EUI_natural_gas', 'PEI_natural_gas'), ('Biomasse', 'EUI_biomass', 'PEI_biomass'), ('Sonstige inkl. Öl', 'EUI_other', 'PEI_other')]:
    f = data[pe_col].div(data[ee_col].where(data[ee_col].ne(0)))
    implicit.append(pd.DataFrame({'Archetype': data.Archetype, 'Träger': carrier, 'f_PE_effektiv': f, 'Bezugsjahr': params['jahr_pe_bewertung']}))
implicit = pd.concat(implicit, ignore_index=True)
display(implicit.groupby('Träger').f_PE_effektiv.agg(['min', 'max']))
primary.to_csv(OUT / 'primaerenergiebilanz.csv', index=False)
implicit.to_csv(OUT / 'pe_faktoren_effektiv.csv', index=False)


## 7. GFZ-Zuordnung, Flächengewichtung und Dichte-Erwartungswert {#sec-gebaeude-gfz}

Die Zuordnung aus `zuordnung_gfz` wird auf die Kategorien der vorhandenen GFZ-Hochrechnung 2021 übersetzt. Die Quellgruppen mit einer bzw. zwei Wohnungen haben identische Zuordnungen und werden **einmal** auf die zusammengefasste GFZ-Gruppe „Ein- und Zweifamilienhäuser“ angewendet. Die 50/50-Aufteilung zwischen EFH_2G und RH_2G wird übernommen. Die vom Auftraggeber am 23.10.2026 bestätigte Regel lautet: **Eine einzige Zielklasse erhält 100 %; die zweite Zielspalte dient nur einer Aufteilung.** Die aktualisierte Excel enthält diese 100-%-Werte nun explizit. Es sind keine fehlenden Quellanteile mehr zu ergänzen. Die Regel bleibt für weitere Einzelzuordnungen implementiert und wird im Audit protokolliert. MFH_2G hat keine Zuordnung und keine positive Fläche; sein Erwartungswert bleibt leer. Es wird kein Ersatz aus anderen Gebäudetypen erfunden.

Aus GFZ-Referenzfläche $R_{u,g,j}$ und Zuordnungsanteil $z_{u,g,k}$ entsteht
$$R_{k,j}=\sum_{u,g}z_{u,g,k}R_{u,g,j},\qquad s_{j|k}=R_{k,j}/\sum_j R_{k,j}.$$
Diese **bedingte Verteilung** wird auf die unabhängig gelieferte Typenfläche übertragen: $A_{k,j}^{2040}=A_k^{2040}s_{j|k}$. Die Referenzfläche wird nicht als zusätzliche nationale Fläche addiert. Die Annahme ist, dass Hüll- und Technikvarianten derselben Geometrie die gleiche Dichteverteilung besitzen; eine räumliche Änderung bis 2040 ist damit noch nicht modelliert. Die vorhandene freigegebene GFZ-Ersatzregel (1,696 % Referenz-BGF) bleibt unverändert.

Als historische **elektrische** Referenz wird die Tabellenparametrisierung der PED-Typen aus Schneider et al. (2023), Gl. 7/Tabelle 5, verwendet [@schneider2023context]:
$$c_D^{el}(d)=-\min\left(\frac{30{,}4}{d+0{,}15}-26{,}4;62\right)\quad[kWh/(m^2\,BGF\,a)].$$
Die Originaltabelle enthält widersprüchliche EE-/PE-Einheitenbeschriftungen (insbesondere bei der Begrenzung). Hier wird die elektrische Tabellenlesart als **Referenzansatz** dokumentiert; daraus entsteht keine neu freigegebene Anforderung. Die bereits PE-gewichteten Kontextwerte der Einzelsimulation werden nicht als EE ausgegeben und nicht zur GFZ-Hochrechnung benutzt. Positive $c_D$ sind virtuelle Gutschriften, negative zusätzliche Anforderungen.

$$\mathbb E_k[GFZ]=\sum_j s_{j|k}d_j,\qquad\mathbb E_k[c_D]=\sum_j s_{j|k}c_D(d_j).$$
Wegen der Nichtlinearität gilt im Allgemeinen $\mathbb E[c_D(d)]\ne c_D(\mathbb E[d])$. NGF-spezifische Kontextwerte ergeben sich durch Division durch $r$, noch **ohne PE-Umrechnung**. Die Verteilung des Kontextwerts wird für jeden GFZ-Bin exportiert; die Typenerwartung beschreibt den Kalibrierungsansatz, nicht einer automatischen Gutschrift für alle Typen.

### Was bedeutet der Erwartungswert?

| Symbol | Bedeutung | Einheit |
|---|---|---|
| $k$ | Energetischer Gebäudetyp; die GFZ-Verteilung folgt seiner Geometrie | – |
| $u,g$ | Statistische Nutzung und Geschoßklasse | – |
| $j$ | GFZ-Bin im Referenzraster | – |
| $d_j$ | GFZ des Bins | m² BGF / m² Grundstück |
| $R_{u,g,j}$ | Referenz-BGF der Nutzung/Geschoßklasse im Bin | m² BGF |
| $z_{u,g,k}$ | Zuordnungsanteil aus `zuordnung_gfz` | 0–1 |
| $s_{j\mid k}$ | Anteil des Bins an der BGF von Typ $k$; Summe über $j$ = 1 | 0–1 |
| $c_D(d_j)$ | Dichtewert im Bin; positiv = Gutschrift, negativ = Zusatzanforderung | kWh EE/(m² BGF·a) |
| $\mathbb E_k[c_D]$ | Mit $s_{j\mid k}$ gewichteter mittlerer Dichtewert des Typs | kWh EE/(m² BGF·a) |
| $A_{BGF,k}$ | Nationale BGF des Typs 2040 | m² BGF |
| $C_{D,k}$ | Virtueller jährlicher Beitrag des Typs: $A_{BGF,k}\mathbb E_k[c_D]$ | kWh EE-Äquivalent/a |

**Beispiel EFH_1G:** Der mittlere Kontextwert beträgt −41,63 kWh EE/(m² BGF·a).
Bei 1 Mio. m² BGF entspricht dies −0,150 PJ-Äquivalent/a. Das negative Vorzeichen
erhöht die Anforderung an diese Geometrie; es beschreibt keine zusätzliche Stromlast.

### Anwendung im Vergleichsfall

Die physische Typenbilanz lautet $b_k=r(e_{PV,k}-e_{el,k})$ je m² BGF.
Der Vergleichsfall ergänzt sie zu $b_{k,D}=b_k+\mathbb E_k[c_D]$.
Damit gilt national:

$$B_D=B_{phys}+C_D,\qquad C_D=\sum_k A_{BGF,k}\mathbb E_k[c_D].$$

Die anschließende Grafik zeigt Bedarf, PV und Kontextwert getrennt für **alle Typen**.
Geometrieauswahl und Achsenregler halten die Darstellung lesbar. Der Tooltip nennt
die nationale BGF; Typen mit Nullfläche tragen national nichts bei. Für MFH_2G bleibt
der nicht zugeordnete Kontextwert leer.

Für den nationalen Vergleich wird die historische Formel auf alle positiv gewichteten
Flächen angewandt. Das ist eine explizite Bewertungsvariante, keine Festlegung, dass
jede Fläche PEQ-berechtigt ist. Eine eingeschränkte Anwendung benötigt typenspezifische
PEQ-Anteile. Die physischen Kennwerte und die PE-Bilanz werden durch diese Variante
nicht verändert.


In [ ]:
mapping = inputs['GFZ_Zuordnung'].copy()
use_map = {'Gebäude für Gemeinschaften': 'Gemeinschaftsgebäude', 'Hotels und ähnliche Gebäude': 'Hotels o.Ä.', 'Bürogebäude': 'Büronutzung', 'Groß- und Einzelhandelsgebäude': 'Groß- und Einzelhandelsgebäude', 'Gebäude des Verkehrs- und Nachrichtenwesens': 'Verkehrs- und Nachrichtenwesen', 'Industrie- und Lagergebäude': 'Industrie und Lagerung', 'Gebäude für Kultur- und Freizeitzwecke sowie des Bildungs- und Gesundheitswesens': 'Kultur, Freizeit, Bildung und Gesundheit', 'Wohngebäude mit einer Wohnung': 'Ein- und Zweifamilienhäuser', 'Wohngebäude mit zwei Wohnungen': 'Ein- und Zweifamilienhäuser', 'Wohngebäude mit drei und mehr Wohnungen': 'Mehrfamilienhaus'}
mapping['GWR_Nutzung'] = mapping.Nutzung.map(use_map)
mapping['GK'] = mapping.Geschoss.str.replace(' Geschoße', '', regex=False).str.replace(' Geschoß', '', regex=False).replace({'6 bis 10': '6-10', '11 und mehr': '11+'})
mapping['Einzelziel_ergänzt'] = mapping.Anteil1.isna() & mapping.Archetype1.notna() & mapping.Archetype2.isna()
if params['single_target_missing_share'] == 1:
    mapping.loc[mapping['Einzelziel_ergänzt'], 'Anteil1'] = 1.0
mapping.to_csv(OUT / 'gfz_zuordnung_audit.csv', index=False)
unique = mapping.drop_duplicates(['GWR_Nutzung', 'GK'])
long = []
for _, row in unique.iterrows():
    for n in [1, 2]:
        if pd.notna(row[f'Archetype{n}']):
            long.append({'GWR_Nutzung': row.GWR_Nutzung, 'GK': row.GK, 'Geometrie': row[f'Archetype{n}'], 'Anteil': row[f'Anteil{n}']})
links = pd.DataFrame(long)
gfz = pd.read_csv(ROOT / 'Berechnungen/gfz_at/outputs/recalculated/gfz_distribution_by_province_use_storeys.csv', dtype={'GK': str})
joined = gfz.merge(links, on=['GWR_Nutzung', 'GK'], how='left', indicator=True)
joined['zugeordnete_BGF'] = joined.BGF_m2 * joined.Anteil
bins = joined.groupby(['Geometrie', 'GFZ'], as_index=False).zugeordnete_BGF.sum()
bins['s'] = bins.zugeordnete_BGF / bins.groupby('Geometrie').zugeordnete_BGF.transform('sum')
bins['c_D_EE_BGF'] = -np.minimum(float(params['density_f_res']) / (bins.GFZ + float(params['density_dx'])) - float(params['density_f_ed']), float(params['density_cutoff']))
bins['c_D_EE_NGF'] = bins.c_D_EE_BGF / r
bins['gew_GFZ'] = bins.GFZ * bins.s
bins['gew_c'] = bins.c_D_EE_BGF * bins.s
density = bins.groupby('Geometrie').agg(GFZ_Mittel=('gew_GFZ', 'sum'), c_D_EE_BGF=('gew_c', 'sum'), Referenz_BGF=('zugeordnete_BGF', 'sum'))
density['c_D_EE_NGF'] = density.c_D_EE_BGF / r
density['c_am_GFZ_Mittel'] = -np.minimum(float(params['density_f_res']) / (density.GFZ_Mittel + float(params['density_dx'])) - float(params['density_f_ed']), float(params['density_cutoff']))
data = data.merge(density[['GFZ_Mittel', 'c_D_EE_BGF', 'c_D_EE_NGF']], left_on='Geometrie', right_index=True, how='left', validate='many_to_one')
print('Nach bestätigter Einzelzielregel ergänzte leere Anteile:', int(mapping['Einzelziel_ergänzt'].sum()), 'Quellzeilen')
affected = unique.loc[unique['Einzelziel_ergänzt'], ['GWR_Nutzung', 'GK']]
affected_area = gfz.merge(affected, on=['GWR_Nutzung', 'GK']).BGF_m2.sum()
print('Davon betroffene Referenz-BGF [%]:', 100 * affected_area / gfz.BGF_m2.sum())
display(density)
matrix = data.loc[data['Nationale BGF 2040'].gt(0), ['Archetype', 'Geometrie', 'Nationale BGF 2040']].merge(bins[['Geometrie', 'GFZ', 's', 'c_D_EE_BGF']], on='Geometrie', validate='many_to_many')
matrix['BGF_2040_m2'] = matrix['Nationale BGF 2040'] * matrix.s
by_type = matrix.groupby('Archetype').BGF_2040_m2.sum()
matrix[['Archetype', 'Geometrie', 'GFZ', 's', 'BGF_2040_m2', 'c_D_EE_BGF']].to_csv(OUT / 'typologie_gfz_2040.csv.gz', index=False, compression='gzip')
bins.to_csv(OUT / 'gfz_kontext_verteilungen.csv', index=False)
density.to_csv(OUT / 'dichte_erwartungswerte.csv')
data[['Archetype', 'Geometrie', 'GFZ_Mittel', 'c_D_EE_BGF', 'c_D_EE_NGF', 'Nationale BGF 2040']].to_csv(OUT / 'typologie_dichte.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, g in bins.groupby('Geometrie'):
    axes[0].plot(g.GFZ, g.s * 100, label=name)
axes[0].set(xlim=(0, 8), xlabel='GFZ', ylabel='BGF-Anteil [% je GFZ-Bin 0,1]')
axes[0].legend(fontsize=7, ncol=2)
density.c_D_EE_BGF.sort_values().plot.barh(ax=axes[1], color='#59856b')
axes[1].axvline(0, color='black', linewidth=0.6)
axes[1].set(xlabel='E[c_D] [kWh EE/(m² BGF a)]', ylabel='')
fig.tight_layout()
fig.savefig(OUT / 'gfz_dichte.png', dpi=180)
plt.show()
data['Bedarf_BGF'] = data.Strombedarf * r
data['PV_BGF'] = data.PV_Erzeugung * r
positive = data['Nationale BGF 2040'].gt(0)
data['Dichtebeitrag_PJ'] = 0.0
data.loc[positive, 'Dichtebeitrag_PJ'] = data.loc[positive, 'Nationale BGF 2040'] * data.loc[positive, 'c_D_EE_BGF'] * 3.6e-09
context_pj = float(data.Dichtebeitrag_PJ.sum())
data[['Archetype', 'Geometrie', 'Nationale BGF 2040', 'Bedarf_BGF', 'PV_BGF', 'c_D_EE_BGF', 'Dichtebeitrag_PJ']].to_csv(OUT / 'typen_bewertung_dichte.csv', index=False)


## 8. Vergleich mit dem nationalen Ziel 2040 {#sec-gebaeude-abgleich}
Dieser Vergleich betrifft ausschließlich den Bestand 2040. Der heutige Bestand wird separat
beschrieben und nicht als zeitgleicher Gegenpart zu einem nationalen Ziel 2040 ausgegeben.
Die gleichen Mobilitätsmengen und dieselbe zentrale Restdeckung werden auf beiden Seiten
angesetzt. Die physische Bilanz bleibt von der virtuellen Dichtebewertung getrennt.


In [ ]:
allocation_path = ROOT / 'Berechnungen/sektorenmodell/outputs/allokation/allokation.json'
allocation = json.loads(allocation_path.read_text(encoding='utf-8'))
top = allocation['result']

def val(metric):
    return float(energy.loc[energy.Stand.eq('2040') & energy['Größe'].eq(metric), 'PJ/a'].iloc[0])
operating = val('Stromsaldo')
extended = operating + top['central_credit_pj'] - top['miv_demand_pj']
comparison = pd.DataFrame([{'Größe': 'PV-Erzeugung', 'Bottom-up [PJ/a]': val('PV_Erzeugung'), 'Top-down [PJ/a]': top['building_pv_pj']}, {'Größe': 'Gebäudestrombedarf', 'Bottom-up [PJ/a]': val('Strombedarf'), 'Top-down [PJ/a]': top['building_pv_pj'] - top['building_operation_pj']}, {'Größe': 'Betrieblicher Stromsaldo', 'Bottom-up [PJ/a]': operating, 'Top-down [PJ/a]': top['building_operation_pj']}, {'Größe': 'Saldo einschließlich MIV', 'Bottom-up [PJ/a]': extended, 'Top-down [PJ/a]': top['building_target_pj']}])
comparison['Differenz [PJ/a]'] = comparison['Bottom-up [PJ/a]'] - comparison['Top-down [PJ/a]']
display(comparison)
comparison.to_csv(OUT / 'abgleich_allokation.csv', index=False)
print('PEQ-Anteile offen:', data.PEQ_Anteil_2040.isna().sum(), 'von', len(data))
sensitivity = pd.DataFrame([{'NGF/BGF': ratio, 'Strombedarf [PJ/a]': val('Strombedarf') * ratio / r, 'Stromsaldo [PJ/a]': operating * ratio / r} for ratio in [0.75, 0.8, 0.85]])
display(sensitivity)
display(comparison)
comparison.to_csv(OUT / 'abgleich_allokation.csv', index=False)


## 9. Übergabe an Dashboard und Bericht
`outputs/analyse/bestand.parquet` enthält beide Szenarien in einer Tabelle: `Szenario`,
`BGF_m2`, vollständige Typenidentität, Merkmale und spezifische Energiekennwerte.
`Szenario` ist entweder `heute` oder `2040`. Das Dashboard wählt genau einen Bestand.
Die PEQ-Auswahl verändert die virtuelle Bewertung, nicht die gelieferten Bestandsflächen.

Zusätzlich werden Tabellen und Grafiken in outputs/analyse erzeugt. Diese neue Analyse
überschreibt den publizierten Stand erst nach fachlicher Durchsicht und bewusster Übernahme.
Bei neuer Lieferung beide Eingabedateien aktualisieren und das Notebook von oben ausführen.


In [ ]:
table = geometry_area.rename(columns={'Nationale BGF heute': 'BGF heute [Mio. m²]', 'Nationale BGF 2040': 'BGF 2040 [Mio. m²]'}).reset_index()
table = table.merge(density[['GFZ_Mittel', 'c_D_EE_BGF']].reset_index(), on='Geometrie', how='left')
(GEN / 'gebaeude-typologien.qmd').write_text(markdown_table(table, 'Gelieferte Flächen und BGF-gewichtete Dichte-Erwartungswerte. Heute: Bezugsjahr offen; Dichtefunktion in elektrischer Energie.', 'tbl-gebaeude-typologien'), encoding='utf-8')
(GEN / 'gebaeude-strom.qmd').write_text(markdown_table(energy.loc[energy.Stand.eq('2040'), ['Größe', 'kWh/(m² NGF a)', 'GWh/a', 'PJ/a']], 'Strombilanz 2040 aus den gelieferten Typenkennwerten bei NGF/BGF = ' + str(r) + '.', 'tbl-gebaeude-strom'), encoding='utf-8')
(GEN / 'gebaeude-endenergie.qmd').write_text(markdown_table(endenergy.loc[endenergy.Stand.eq('2040')].drop(columns='Stand'), 'Endenergie 2040 nach Energieträgern. Strom enthält PV-Direktverbrauch; sonstige thermische Energie wird separat geführt.', 'tbl-gebaeude-endenergie'), encoding='utf-8')
(GEN / 'gebaeude-abgleich.qmd').write_text(markdown_table(comparison, 'Vergleich bei gleichen elektrischen Bilanzgrenzen; Top-down-Zuordnungen gemäß dokumentiertem Referenzfall.', 'tbl-gebaeude-abgleich'), encoding='utf-8')
(GEN / 'gebaeude-status.qmd').write_text(markdown_table(status, 'Flächengewichtete Zuordnung zu Bestand, Neubau und Sanierung im Zieljahr.', 'tbl-gebaeude-status'), encoding='utf-8')
(GEN / 'gebaeude-pe.qmd').write_text(markdown_table(primary.loc[primary.Stand.eq('2040')].drop(columns='Stand'), 'Gelieferter PE-Bewertungsstand; Faktorversion und Bezugsjahr noch zu bestätigen, ohne virtuelle Kontextwerte.', 'tbl-gebaeude-pe'), encoding='utf-8')
summary = {'provenance': provenance, 'allocation_sha256': sha256(allocation_path), 'ngf_bgf': r, 'type_count': len(data), 'positive_types_2040': int(data['Nationale BGF 2040'].gt(0).sum()), 'bgf_2040_m2': float(data['Nationale BGF 2040'].sum()), 'bgf_today_m2': float(data['Nationale BGF heute'].sum()), 'electricity_demand_pj': val('Strombedarf'), 'pv_pj': val('PV_Erzeugung'), 'electricity_balance_pj': operating, 'extended_balance_pj': extended, 'gap_topdown_pj': extended - top['building_target_pj'], 'single_target_missing_share_rows': int(mapping['Einzelziel_ergänzt'].sum()), 'affected_reference_area_pct': 100 * affected_area / gfz.BGF_m2.sum(), 'pe_balance_pj': float(primary.loc[primary.Stand.eq('2040') & primary['Größe'].eq('PE_Saldo'), 'PJ_PE/a'].iloc[0]), 'simulated_type_count': original_count, 'synthetic_type_count': 0, 'controls': 'All assertions passed; source values are not an independently rerun PEExcel simulation.'}
summary['pe_balance_pj'] = summary['pe_balance_pj'] if np.isfinite(summary['pe_balance_pj']) else None
(OUT / 'validation.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
data.to_csv(OUT / 'typen_ergebnisse.csv', index=False)
area_summary.to_csv(OUT / 'flaechenbilanz.csv', index=False)
status.to_csv(OUT / 'zustandsflaechen_2040.csv', index=False)
print(json.dumps(summary, ensure_ascii=False, indent=2))
bestandsliste = []
for scenario in ['heute', '2040']:
    bestand = data.copy()
    bestand['Szenario'] = scenario
    bestand['BGF_m2'] = bestand['Nationale BGF ' + scenario]
    bestandsliste.append(bestand)
bestand = pd.concat(bestandsliste, ignore_index=True)
bestand.to_parquet(OUT / 'bestand.parquet', index=False)
display(bestand.groupby('Szenario').BGF_m2.sum().rename('Nationale BGF [m²]'))
